# Olist Insights - Análise para Colab

Este notebook carrega os dados do Olist e gera insights em torno de pedidos, clientes, pagamentos, avaliações, produtos, logística e geolocalização.
O objetivo é identificar dores e oportunidades que podem ser transformadas em agentes de IA e automação.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings("ignore")


## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Carregar os dados

Vamos importar as tabelas principais do Olist que serão usadas nas análises.


In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
customers = pd.read_csv(base_path + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(base_path + 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
product_category = pd.read_csv(base_path + 'product_category_name_translation.csv')

print('Dados carregados com sucesso')
print('Shapes:')
print('customers', customers.shape)
print('geolocation', geolocation.shape)
print('order_items', order_items.shape)
print('payments', payments.shape)
print('reviews', reviews.shape)
print('orders', orders.shape)
print('products', products.shape)
print('sellers', sellers.shape)
print('product_category', product_category.shape)


## Inspeção inicial dos dados

Vamos revisar colunas, tipos e valores ausentes para entender a qualidade dos dados antes das análises.


In [ ]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'product_category': product_category,
}
for name, df in datasets.items():
    print('---', name, '---')
    print(df.dtypes)
    print('missing:', df.isna().sum().sort_values(ascending=False).head(10).to_dict())
    print()


## Criar uma tabela analítica mestre

Vamos unir clientes, pedidos, itens, produtos, pagamentos e avaliações para facilitar cruzamentos de análise.


In [ ]:
df_orders_customers = pd.merge(orders, customers, on='customer_id', how='inner')
df_items_products = pd.merge(order_items, products, on='product_id', how='inner')
df_items_products_sellers = pd.merge(df_items_products, sellers, on='seller_id', how='inner')
df_master = pd.merge(df_orders_customers, df_items_products, on='order_id', how='inner')
df_master = pd.merge(df_master, payments, on='order_id', how='left')
df_master = pd.merge(df_master, reviews, on='order_id', how='left')

if 'product_category_name' in df_master.columns and 'product_category_name_english' in product_category.columns:
    df_master = pd.merge(df_master, product_category, on='product_category_name', how='left')

print('Master shape:', df_master.shape)
df_master.head()


### Insight 1: Top 10 categorias por receita

Este gráfico mostra quais categorias de produto estão gerando mais receita no conjunto de pedidos.
Este é um bom ponto de partida para identificar prioridades de recomendação, otimização de estoque e automação de produto.


In [ ]:
top_categories = df_master.groupby('product_category_name')['price'].sum().reset_index()
top_categories = top_categories.sort_values('price', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, x='price', y='product_category_name', palette='viridis')
plt.title('Top 10 categorias por receita total')
plt.xlabel('Receita total (R$)')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()


### Insight 2: Volume de pedidos por estado do cliente

Entender a distribuição geográfica de pedidos ajuda a identificar regiões com maior demanda e possíveis desafios logísticos.


In [ ]:
orders_by_state = df_orders_customers['customer_state'].value_counts().reset_index()
orders_by_state.columns = ['state', 'order_count']

plt.figure(figsize=(12, 6))
sns.barplot(data=orders_by_state, x='order_count', y='state', palette='magma')
plt.title('Volume de pedidos por estado do cliente')
plt.xlabel('Quantidade de pedidos')
plt.ylabel('Estado')
plt.tight_layout()
plt.show()


### Insight 3: Status de pedidos e risco operacional

Analisar os status de pedidos revela onde há maior potencial de gargalos operacionais e atrasos que podem afetar a experiência do cliente.


In [ ]:
orders_status = orders['order_status'].value_counts().reset_index()
orders_status.columns = ['order_status', 'count']

plt.figure(figsize=(10, 5))
sns.barplot(data=orders_status, x='count', y='order_status', palette='coolwarm')
plt.title('Quantidade de pedidos por status')
plt.xlabel('Quantidade')
plt.ylabel('Status')
plt.tight_layout()
plt.show()


### Insight 4: Pagamentos por tipo e valor

Este gráfico revela como diferentes tipos de pagamento se comportam em relação ao valor gasto, ajudando a detectar padrões de risco e preferência.


In [ ]:
df_payments_orders = pd.merge(payments, orders, on='order_id', how='inner')

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_payments_orders, x='payment_type', y='payment_value', showfliers=False)
plt.title('Distribuição do valor dos pagamentos por tipo')
plt.xlabel('Tipo de pagamento')
plt.ylabel('Valor do pagamento (R$)')
plt.tight_layout()
plt.show()


### Insight 5: Tempo logístico entre aprovação e entrega

Avaliar prazos de entrega ajuda a identificar atrasos e oportunidades para agentes que alertem problemas de logística e melhorem a experiência do cliente.


In [ ]:
orders_time = orders.copy()
orders_time['order_approved_at'] = pd.to_datetime(orders_time['order_approved_at'])
orders_time['order_delivered_carrier_date'] = pd.to_datetime(orders_time['order_delivered_carrier_date'])
orders_time['order_delivered_customer_date'] = pd.to_datetime(orders_time['order_delivered_customer_date'])
orders_time['time_to_carrier'] = (orders_time['order_delivered_carrier_date'] - orders_time['order_approved_at']).dt.days
orders_time['time_to_customer'] = (orders_time['order_delivered_customer_date'] - orders_time['order_approved_at']).dt.days

plt.figure(figsize=(12, 5))
sns.histplot(orders_time['time_to_customer'].dropna(), bins=40, kde=True)
plt.title('Distribuição do tempo de entrega ao cliente (dias)')
plt.xlabel('Dias')
plt.tight_layout()
plt.show()

orders_time[['time_to_carrier', 'time_to_customer']].describe()


### Insight 6: Avaliações dos clientes

Entender como os clientes avaliam a experiência permite preparar agentes de análise de sentimento e suporte proativo.


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=reviews, x='review_score', palette='crest')
plt.title('Distribuição das avaliações dos pedidos')
plt.xlabel('Nota da avaliação')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
reviews['comment_length'] = reviews['review_comment_message'].str.len()
plt.figure(figsize=(10, 5))
sns.histplot(reviews['comment_length'], bins=40, kde=True)
plt.title('Comprimento dos comentários de avaliação')
plt.xlabel('Número de caracteres')
plt.tight_layout()
plt.show()


### Insight 7: Principais vendedores por receita

Identificar os sellers que mais contribuem para a receita é útil para ações de parceria e monitoramento de desempenho.


In [ ]:
seller_revenue = df_master.groupby('seller_id')['price'].sum().reset_index()
top_sellers = seller_revenue.sort_values('price', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_sellers, x='price', y='seller_id', palette='coolwarm')
plt.title('Top 10 vendedores por receita')
plt.xlabel('Receita total (R$)')
plt.ylabel('Seller ID')
plt.tight_layout()
plt.show()


## Conclusões e oportunidades para agentes de IA

A partir desses levantamentos, as oportunidades mais relevantes incluem:
- agentes de monitoramento de atrasos logísticos,
- assistentes para análise de reviews e detecção de satisfação,
- recomendações de categorias e cross-sell com base nas top receitas,
- detecção de desvios em pagamentos e preferências por tipo de pagamento,
- dashboards regionais para apoiar a logística e a expansão de cobertura.
